# 💊 Pfizer Pharmaceutical Document RAG Chatbot
### Enhanced Multi-Document Q&A with Intelligent RAG Pipeline

This notebook combines:
- **Mistral-7B-Instruct** (GGUF via llama-cpp) as the answer-generation LLM
- **Gemini** for document classification & boundary detection (fast API calls)
- **Project 7's** logical page grouping + page-boundary detection + rich metadata chunking
- **The Enhanced Q&A interface**: 3-column Gradio UI, document info panel, retrieval settings, status bar
- **Pfizer branding** with chat export

**Colab Pro GPU recommended** — Mistral loads fully on GPU with `n_gpu_layers=-1`


In [1]:
# ============================================================
# STEP 1: Install Required Packages
# ============================================================
# llama-cpp-python built with CUDA so Mistral runs on the Colab GPU.
# Everything else is standard — LlamaIndex, Gradio, PyMuPDF, etc.

# Install llama-cpp-python with CUDA backend (required for GPU offloading)
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install -q llama-cpp-python --upgrade --force-reinstall --no-cache-dir

# Core packages
!pip install -q gradio
!pip install -q pymupdf PyPDF2
!pip install -q sentence-transformers
!apt-get install -q tesseract-ocr
!pip install -q pytesseract pillow

# LlamaIndex ecosystem
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-llms-llama-cpp
!pip install -q llama-index-llms-gemini
!pip install -q google-generativeai

!pip install -q numpy pandas

print("✅ All packages installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 343.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 556.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 822.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 332.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 624.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)
  

In [2]:
# ============================================================
# STEP 2: Imports and Global Configuration
# ============================================================
import os, json, uuid, time
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

import fitz          # PyMuPDF
import numpy as np
import pandas as pd
import gradio as gr
import pytesseract
from PIL import Image
import io
from concurrent.futures import ThreadPoolExecutor, as_completed

# LlamaIndex
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.schema import TextNode
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import (
    MetadataFilters, MetadataFilter, FilterOperator
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.llama_cpp import LlamaCPP
from llama_cpp import Llama

# Gemini (for classification & boundary detection)
import google.generativeai as genai
from google.colab import userdata


# ── Configure Gemini (classification only) ────────────────────────────────
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
_gemini_model = genai.GenerativeModel("models/gemini-2.5-flash")

# ── Shared embedding model ────────────────────────────────────────────────
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.llm = None   # We handle LLM calls manually

print("✅ Imports and configuration complete.")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM is explicitly disabled. Using MockLLM.
✅ Imports and configuration complete.


In [3]:
# ============================================================
# STEP 3: Load Mistral-7B-Instruct via LlamaCPP
# ============================================================

MISTRAL_PATH = "/content/mistral.gguf"

# Max context tokens the model is loaded with.
# Mistral-7B-Instruct-v0.2 supports up to 32768.
# 16384 is a safe ceiling on a Colab GPU (T4/A100).
MISTRAL_N_CTX     = 16384
MISTRAL_MAX_REPLY = 512    # tokens reserved for the answer
MISTRAL_CTX_GUARD = 0.90  # warn/trim when prompt uses >90% of ctx budget

def load_mistral(model_path: str = MISTRAL_PATH, n_gpu_layers: int = -1):
    """Download (if needed) and load Mistral-7B-Instruct Q4_K_M GGUF."""
    if not os.path.exists(model_path):
        print("⬇️  Downloading Mistral-7B-Instruct-v0.2 Q4_K_M (~4.4 GB) ...")
        os.system(
            f"wget -q https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
            f"/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}"
        )
        print("✅ Download complete.")

    size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"🤖 Loading Mistral ({size_mb:.0f} MB) | n_ctx={MISTRAL_N_CTX} | n_gpu_layers={n_gpu_layers} ...")

    llm = Llama(
        model_path=model_path,
        n_gpu_layers=n_gpu_layers,
        n_ctx=MISTRAL_N_CTX,
        verbose=False,
    )
    print(f"✅ Mistral loaded. Context window: {MISTRAL_N_CTX} tokens.")
    return llm


mistral_llm = load_mistral()


# ── Token estimation ──────────────────────────────────────────────────────────
def estimate_tokens(text: str) -> int:
    """Fast approximation: ~1 token per 3.5 characters for English text."""
    return int(len(text) / 3.5)


def available_prompt_budget() -> int:
    """Max tokens the prompt can use, leaving room for the reply."""
    return MISTRAL_N_CTX - MISTRAL_MAX_REPLY


# ── Non-streaming call (used by generate_answer) ──────────────────────────────
def call_mistral(prompt: str) -> str:
    """Call Mistral synchronously and return plain text response."""
    output = mistral_llm.create_completion(
        prompt=prompt,
        max_tokens=MISTRAL_MAX_REPLY,
        temperature=0.1,
        stream=False,
    )
    return output["choices"][0]["text"].strip()


# ── Streaming call (used by Gradio chat handler) ──────────────────────────────
def stream_mistral(prompt: str):
    """Stream token text from Mistral for real-time Gradio display."""
    stream = mistral_llm.create_completion(
        prompt=prompt,
        max_tokens=MISTRAL_MAX_REPLY,
        temperature=0.1,
        stream=True,
    )
    for chunk in stream:
        text = chunk["choices"][0].get("text", "")
        if text:
            yield text


def call_gemini(prompt: str) -> str:
    """Wrapper: call Gemini Flash and return plain text response."""
    try:
        response = _gemini_model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        print(f"Gemini error: {e}")
        return ""

⬇️  Downloading Mistral-7B-Instruct-v0.2 Q4_K_M (~4.4 GB) ...
✅ Download complete.
🤖 Loading Mistral (4166 MB) | n_ctx=16384 | n_gpu_layers=-1 ...


llama_context: n_ctx_seq (16384) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Mistral loaded. Context window: 16384 tokens.


In [4]:
# ============================================================
# STEP 4: Data Structures
# ============================================================
# These dataclasses track pages → logical documents → chunks
# so every chunk carries full provenance (doc type, page range).
# Adapted from the Enhanced Q&A notebook with Project 7 metadata fields.

@dataclass
class PageInfo:
    """One physical page extracted from the PDF."""
    page_num: int
    text: str
    doc_type: str = "Other"
    page_in_doc: int = 1      # 1-indexed position within logical doc
    is_new_doc: bool = False   # True when this page starts a new logical doc

@dataclass
class LogicalDocument:
    """A group of consecutive pages that form one pharmaceutical sub-document."""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    source_file: str = "Unknown"  # <-- ADDED
    chunks: List[Dict] = field(default_factory=list)

@dataclass
class ChunkMetadata:
    """Rich metadata attached to every text chunk stored in the vector index."""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    source_file: str = "Unknown"  # <-- ADDED




In [5]:
# ============================================================
# STEP 5: Document Intelligence Functions
# ============================================================
# Uses Gemini for classification/boundary (fast API, structured prompts)
# Uses the enhanced JSON-output classifier from Project 7.

VALID_DOC_TYPES = [
    "Cover Letter", "Certificate of Quality", "Packaging Specification",
    "BSE/TSE Declaration", "Material Description", "Supplier Qualification",
    "Chain of Custody", "Other"
]

# ── Rule-based keyword classifier (zero API calls) ────────────────────────────
# Covers the majority of pharmaceutical doc types unambiguously.
# Returns None when uncertain so Gemini handles the remainder.

RULE_PATTERNS: Dict[str, List[str]] = {
    "Cover Letter": [
        "to whom it may concern", "this letter is provided",
        "storage temperature", "operating temperature", "sincerely",
        "recommended storage"
    ],
    "Certificate of Quality": [
        "certificate of quality", "certificate of analysis",
        "lot number", "date of manufacture", "expiration date",
        "release criteria", "conforms", "purity by"
    ],
    "Packaging Specification": [
        "packaging specification", "packaging component", "blister tray",
        "lid film", "secondary carton", "ecn number", "change history",
        "drawing reference", "effective date"
    ],
    "BSE/TSE Declaration": [
        "transmissible spongiform", "bse", "tse", "animal origin",
        "bovine", "encephalopathies", "prion"
    ],
    "Material Description": [
        "material description", "materials of construction",
        "sterilization compatibility", "operating pressure",
        "physical properties", "shelf life", "platinum-cured"
    ],
    "Supplier Qualification": [
        "supplier qualification", "supplier name", "supplier code",
        "on-site audit", "iso 9001", "iso 13485",
        "fda establishment", "qualification status", "quality agreement"
    ],
    "Chain of Custody": [
        "chain of custody", "manufactured at", "traceability",
        "lot traceable", "distribution center", "chain of custody"
    ],
}

def rule_based_classify(text: str) -> Optional[str]:
    """
    Fast keyword-based classification — no API call.
    Returns a doc type string if confident, None if uncertain.
    """
    text_lower = text[:1500].lower()
    scores: Dict[str, int] = {}

    for doc_type, patterns in RULE_PATTERNS.items():
        score = sum(1 for p in patterns if p in text_lower)
        if score > 0:
            scores[doc_type] = score

    if not scores:
        return None

    best      = max(scores, key=scores.get)
    best_score = scores[best]

    # Trust single-match for highly specific types; require 2+ for generic ones
    specific_types = {"BSE/TSE Declaration", "Chain of Custody", "Packaging Specification"}
    threshold = 1 if best in specific_types else 2
    return best if best_score >= threshold else None

def clean_doc_type(raw: str) -> str:
    """Normalise an LLM response into one of the VALID_DOC_TYPES labels."""
    cleaned = raw.strip().lower().replace('"','').replace('`','').replace('*','').replace('.','')
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned:
            return label
    return "Other"


def classify_doc_type(text: str, max_chars: int = 1500) -> str:
    """
    Classify a page's document type using Gemini + JSON-structured prompt.
    Adapted from Project 7's enhanced classify_doc_type_llm().
    """
    sample = text[:max_chars]
    prompt = f"""You are an expert pharmaceutical document classifier.
    Classify the page below into EXACTLY ONE type. Output ONLY valid JSON.

    PERMITTED TYPES:
    - "Cover Letter": Formal letter (often "To Whom It May Concern") about product info or storage.
    - "Certificate of Quality": Contains lot numbers, manufacture/expiry dates, test results.
    - "Packaging Specification": Packaging components, materials, part numbers, change history.
    - "BSE/TSE Declaration": Animal-origin material declarations, TSE compliance.
    - "Material Description": Materials of construction, sterilization compatibility, physical properties.
    - "Supplier Qualification": Supplier audits, ISO 9001/13485 certifications, approved products.
    - "Chain of Custody": Manufactured assemblies, traceability, shipment flow.
    - "Other": Only if none of the above fit.

    Page Content:
    {sample}

    Output ONLY: {{"reasoning": "<one sentence>", "document_type": "<type>"}}
    """
    raw = call_gemini(prompt)
    try:
        start, end = raw.find('{'), raw.rfind('}') + 1
        parsed = json.loads(raw[start:end])
        return clean_doc_type(parsed.get("document_type", "Other"))
    except Exception:
        return clean_doc_type(raw)


def is_same_document(prev_text: str, curr_text: str, current_doc_type: str = None) -> bool:
    """
    Detect whether two consecutive pages belong to the same logical document.
    Uses boundary detection prompt from Project 7.
    Returns True if same document.
    """
    prev_sample = prev_text[-600:] if len(prev_text) > 600 else prev_text
    curr_sample = curr_text[:600] if len(curr_text) > 600 else curr_text

    prompt = f"""Determine if these two consecutive pages are from the SAME pharmaceutical document.

    Current document type: {current_doc_type or 'Unknown'}

    A NEW document starts when the page has:
    - A different title/heading (e.g., "Certificate of Quality" vs "Packaging Specification")
    - A completely different topic or subject matter
    - Its own header with a new document number or reference

    Pages belong to the SAME document when:
    - The second page says "continued" or "page 2 of 2"
    - The content directly continues the previous page's discussion
    - They share the same document number or title

    End of Previous Page:
    ...{prev_sample}

    Start of Current Page:
    {curr_sample}...

    Answer ONLY 'Yes' (same document) or 'No' (different document)."""
    try:
        response = call_gemini(prompt)
        return response.strip().lower().startswith('yes')
    except Exception as e:
        print(f"  ⚠️  Boundary detection error: {e}")
        return True   # default: keep together if uncertain


def predict_doc_type_for_query(query: str) -> Tuple[str, float]:
    """
    Route a user query to the most likely document type.
    Returns (predicted_type, confidence).
    Adapted from Project 7 + Enhanced Q&A notebook.
    """
    prompt = f"""Analyze this query and predict which pharmaceutical document type
    would most likely contain the answer.

    Query: "{query}"

    Choose the MOST LIKELY type from:
    - Cover Letter: Formal letters about product information or storage conditions
    - Certificate of Quality: Lot numbers, manufacture/expiration dates, test results
    - Packaging Specification: Packaging components, materials, part numbers
    - BSE/TSE Declaration: Animal-origin material declarations, TSE compliance
    - Material Description: Materials of construction, sterilization compatibility
    - Supplier Qualification: Supplier audits, ISO certifications, approved products
    - Chain of Custody: Manufactured assemblies, traceability, shipment flow
    - Other: General or unclear queries OR queries asking to compare/search ACROSS multiple documents.

    CRITICAL RULE: If the user query uses phrases like "across these documents", "compare",
    or asks for a general summary of everything, you MUST select "Other" so the system knows to search globally.

    Respond in JSON: {{"type": "DocumentType", "confidence": 0.85}}"""
    try:
        raw = call_gemini(prompt)
        start, end = raw.find('{'), raw.rfind('}') + 1
        parsed = json.loads(raw[start:end])
        predicted = clean_doc_type(parsed.get("type", "Other"))
        confidence = float(parsed.get("confidence", 0.5))
        return predicted, confidence
    except Exception as e:
        print(f"  ⚠️  Query routing error: {e}")
        return "Other", 0.0


In [13]:
# ============================================================
# STEP 6: PDF Extraction and Logical Document Grouping
# ============================================================
# Combines:
#  - PyMuPDF text extraction (Enhanced Q&A notebook)
#  - Page-level boundary detection and metadata (Project 7)
#  - Logical grouping of consecutive same-document pages (Project 7)

def extract_pages(pdf_file) -> List[PageInfo]:
    """Extract raw text from every page using PyMuPDF, with OCR fallback for scanned pages."""
    if isinstance(pdf_file, dict) and "content" in pdf_file:
        doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
    elif hasattr(pdf_file, "read"):
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_file)

    pages = []
    scanned_count = 0

    for i, page in enumerate(doc):
        text = page.get_text().strip()

        if len(text) < 50:
            try:
                pix = page.get_pixmap(dpi=300)
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
                scanned_count += 1
                print(f"  🔍 Page {i+1}: OCR extracted {len(text)} characters")
            except Exception as e:
                print(f"  ⚠️  Page {i+1}: OCR failed — {e}")
                text = ""

        pages.append(PageInfo(page_num=i + 1, text=text))

    doc.close()

    if scanned_count > 0:
        print(f"  📷 {scanned_count} scanned page(s) processed via OCR")
    print(f"  📄 Extracted {len(pages)} pages from PDF")
    return pages


# def detect_boundaries_and_classify(pages: List[PageInfo]) -> List[PageInfo]:
#     """
#     Walk through pages, classify each, and mark document boundaries.
#     Uses the dual-check strategy from Project 7:
#       1. Boundary detection says 'new document'
#       2. Classification confirms the type actually changed
#     """
#     current_doc_type = None
#     page_in_doc = 1

#     for i, page in enumerate(pages):
#         if i == 0:
#             current_doc_type = classify_doc_type(page.text)
#             page.doc_type = current_doc_type
#             page.is_new_doc = True
#             page.page_in_doc = 1
#             print(f"  Page {page.page_num}: [NEW] {current_doc_type}")
#         else:
#             prev_text = pages[i - 1].text
#             same = is_same_document(prev_text, page.text, current_doc_type)

#             if not same:
#                 # Confirm with a second classification call
#                 new_type = classify_doc_type(page.text)
#                 if new_type != current_doc_type:
#                     # Genuinely a new document
#                     current_doc_type = new_type
#                     page_in_doc = 1
#                     page.is_new_doc = True
#                     print(f"  Page {page.page_num}: [NEW] {current_doc_type}")
#                 else:
#                     # Same type — treat as continuation
#                     page_in_doc += 1
#                     page.is_new_doc = False
#             else:
#                 page_in_doc += 1
#                 page.is_new_doc = False

#             page.doc_type = current_doc_type
#             page.page_in_doc = page_in_doc

#     return pages


def group_pages_into_logical_docs(pages: List[PageInfo], filename: str = "Unknown") -> List[LogicalDocument]:
    """
    Combine consecutive pages that share a document into one LogicalDocument.
    Direct port of group_pages() from Project 7.
    """
    logical_docs = []
    current = {"text": "", "doc_type": None, "page_start": None, "doc_id": None}

    for page in pages:
        if page.is_new_doc and current["text"]:
            logical_docs.append(LogicalDocument(
                doc_id=current["doc_id"],
                doc_type=current["doc_type"],
                page_start=current["page_start"],
                page_end=page.page_num - 1,
                text=current["text"].strip(),
                source_file=filename  # <-- ADDED
            ))
            current = {"text": "", "doc_type": None, "page_start": None, "doc_id": None}

        if current["doc_id"] is None:
            # FIX: Create globally unique IDs using the filename to prevent overwriting
            safe_fname = "".join(c if c.isalnum() else "_" for c in filename)
            current["doc_id"] = f"{safe_fname}_doc_{len(logical_docs)}"
            current["page_start"] = page.page_num

        current["text"] += "\n\n" + page.text
        current["doc_type"] = page.doc_type

    # Flush the last document
    if current["text"]:
        logical_docs.append(LogicalDocument(
            doc_id=current["doc_id"],
            doc_type=current["doc_type"],
            page_start=current["page_start"],
            page_end=pages[-1].page_num,
            text=current["text"].strip(),
            source_file=filename  # <-- ADDED
        ))

    print(f"  📂 Identified {len(logical_docs)} logical document(s):")
    for ld in logical_docs:
        print(f"     • {ld.doc_type} (pages {ld.page_start}–{ld.page_end})")

    return logical_docs


def classify_all_pages_parallel(pages: List[PageInfo], max_workers: int = 5) -> List[str]:
    """
    Two-pass classifier:
      Pass 1 — instant rule-based classification (no API calls)
      Pass 2 — Gemini in parallel for pages rules couldn't resolve
    """
    results      = [None] * len(pages)
    needs_gemini = []

    # Pass 1: rules
    for i, page in enumerate(pages):
        result = rule_based_classify(page.text)
        if result:
            results[i] = result
        else:
            needs_gemini.append(i)

    rule_hits = len(pages) - len(needs_gemini)
    print(f"  ⚡ Rule-based: {rule_hits}/{len(pages)} pages classified instantly")

    # Pass 2: Gemini in parallel for the remainder
    if needs_gemini:
        print(f"  🌐 Gemini classifying {len(needs_gemini)} uncertain page(s) in parallel...")
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_idx = {
                executor.submit(classify_doc_type, pages[i].text): i
                for i in needs_gemini
            }
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    results[idx] = future.result()
                except Exception:
                    results[idx] = "Other"

    return results


def detect_boundaries_parallel(pages: List[PageInfo], doc_types: List[str], max_workers: int = 5) -> List[bool]:
    """
    Determine document boundaries for all page transitions.

    Optimization:
    - If adjacent pages have DIFFERENT classified types → boundary confirmed, no API call
    - If adjacent pages have the SAME type → call is_same_document only then (in parallel)

    Returns a list of booleans: is_new_doc[i] for pages[i].
    """
    n          = len(pages)
    is_new_doc = [False] * n
    is_new_doc[0] = True   # first page always starts a new document

    type_changed  = []   # indices where type already changed — instant boundary
    needs_check   = []   # indices where type is same — need is_same_document

    for i in range(1, n):
        if doc_types[i] != doc_types[i - 1]:
            type_changed.append(i)
        else:
            needs_check.append(i)

    print(f"  ✂️  Type-change boundaries (no API): {len(type_changed)}")
    print(f"  🔍 Same-type transitions to check (parallel): {len(needs_check)}")

    # Mark type-change boundaries instantly
    for i in type_changed:
        is_new_doc[i] = True

    # Resolve same-type transitions in parallel
    if needs_check:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_idx = {
                executor.submit(
                    is_same_document,
                    pages[i - 1].text,
                    pages[i].text,
                    doc_types[i]
                ): i
                for i in needs_check
            }
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    same = future.result()
                    is_new_doc[idx] = not same
                except Exception:
                    is_new_doc[idx] = False   # default: continuation

    return is_new_doc


def detect_boundaries_and_classify(pages: List[PageInfo]) -> List[PageInfo]:
    """
    Optimized boundary detection and classification.
    Replaces the old sequential version.
    """
    # Step 1: classify all pages (rules + parallel Gemini)
    doc_types = classify_all_pages_parallel(pages)

    # Step 2: detect boundaries (type-change fast path + parallel same-type check)
    is_new = detect_boundaries_parallel(pages, doc_types)

    # Step 3: apply results
    page_in_doc = 1
    for i, page in enumerate(pages):
        page.doc_type   = doc_types[i]
        page.is_new_doc = is_new[i]
        if is_new[i]:
            page_in_doc = 1
            print(f"  Page {page.page_num}: [NEW] {page.doc_type}")
        else:
            page_in_doc += 1
        page.page_in_doc = page_in_doc

    return pages


def extract_and_analyze_pdf(pdf_file, filename: str = "Unknown") -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """Full pipeline: extract → classify (parallel) → detect boundaries (parallel) → group."""
    print(f"🔍 Starting PDF extraction and analysis for {filename}...")
    pages = extract_pages(pdf_file)

    print("🧠 Classifying pages and detecting document boundaries...")
    pages = detect_boundaries_and_classify(pages)

    logical_docs = group_pages_into_logical_docs(pages, filename)
    return pages, logical_docs

In [7]:
# ============================================================
# STEP 7: Chunking with Rich Metadata Preservation
# ============================================================
# Converts each LogicalDocument into overlapping text chunks,
# attaching full provenance metadata to every chunk.
# Strategy from Project 7 + Enhanced Q&A notebook (overlapping windows).


def chunk_logical_document(
    logical_doc: LogicalDocument,
    chunk_size: int = 500,
    chunk_overlap: int = 100
) -> List[ChunkMetadata]:
    """
    Sliding-window word chunker with overlap.
    Returns ChunkMetadata objects (not yet LlamaIndex Documents).
    """
    words = logical_doc.text.split()
    chunks = []

    if len(words) <= chunk_size:
        chunks.append(ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_0",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=0,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=logical_doc.text,
            source_file=logical_doc.source_file # <-- ADDED
        ))
    else:
        stride = chunk_size - chunk_overlap
        total_words = len(words)

        for i, start in enumerate(range(0, total_words, stride)):
            end = min(start + chunk_size, total_words)
            chunk_text = ' '.join(words[start:end])

            frac = start / total_words
            page_span = logical_doc.page_end - logical_doc.page_start
            chunk_page_start = logical_doc.page_start + int(frac * page_span)
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)

            chunks.append(ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=i,
                page_start=chunk_page_start,
                page_end=chunk_page_end,
                text=chunk_text,
                source_file=logical_doc.source_file # <-- ADDED
            ))

            if end >= total_words:
                break

    return chunks

def chunks_to_llama_documents(chunks: List[ChunkMetadata]) -> List[Document]:
    """
    Convert ChunkMetadata objects into LlamaIndex Document objects
    with the metadata dict that MetadataFilters will query against.
    """
    docs = []
    for c in chunks:
        docs.append(Document(
            text=c.text,
            metadata={
                "chunk_id":    c.chunk_id,
                "doc_id":      c.doc_id,
                "doc_type":    c.doc_type,
                "chunk_index": c.chunk_index,
                "page_start":  c.page_start,
                "page_end":    c.page_end,
                "source_file": c.source_file  # <-- FIXED HARDCODING
            },
            id_=c.chunk_id
        ))
    return docs


def process_all_documents(
    logical_docs: List[LogicalDocument],
    chunk_size: int = 500,
    chunk_overlap: int = 100
) -> Tuple[List[ChunkMetadata], List[Document]]:
    """
    Chunk all logical documents; return (ChunkMetadata list, LlamaIndex Document list).
    """
    all_chunk_meta = []
    for ld in logical_docs:
        chunks = chunk_logical_document(ld, chunk_size, chunk_overlap)
        ld.chunks = chunks
        all_chunk_meta.extend(chunks)
        print(f"  ✂️  {ld.doc_type}: {len(chunks)} chunk(s)")

    llama_docs = chunks_to_llama_documents(all_chunk_meta)
    print(f"  ✅ Total chunks: {len(all_chunk_meta)}")
    return all_chunk_meta, llama_docs


In [8]:
# ============================================================
# STEP 8: LlamaIndex Indexing and Metadata-Filtered Retrieval
# ============================================================
# Builds a VectorStoreIndex from all LlamaIndex Documents.
# Retrieval uses MetadataFilters to scope search to the predicted doc type.
# Based on Project 7's retrieve_nodes() + Enhanced Q&A's query routing.

def build_vector_index(llama_docs: List[Document]) -> VectorStoreIndex:
    """Build (or rebuild) the VectorStoreIndex from chunked LlamaIndex Documents."""
    print("🗂️  Building vector index...")
    index = VectorStoreIndex.from_documents(
        llama_docs,
        embed_model=Settings.embed_model,
        show_progress=True
    )
    print(f"  ✅ Index ready with {len(llama_docs)} nodes.")
    return index


def retrieve_chunks(
    index: VectorStoreIndex,
    query: str,
    filter_doc_type: Optional[str] = None,
    auto_route: bool = True,
    k: int = 4,
    *,
    route_cache: dict = None,
    min_confidence_to_filter: float = 0.70,
) -> Tuple[List, Optional[str], float]:
    """
    Retrieve relevant chunks with optional metadata filtering and auto-routing.

    Improvements:
    - Cache Gemini routing decisions (query -> (type, confidence))
    - Correctly handle routed_type == "Other" (treat as global, but not "low confidence")
    - Avoid Gemini calls if an explicit filter is provided
    """

    if route_cache is None:
        # attach a cache to the function object if none passed (simple in-notebook persistence)
        if not hasattr(retrieve_chunks, "_route_cache"):
            retrieve_chunks._route_cache = {}
        route_cache = retrieve_chunks._route_cache

    routed_type = None
    confidence = 1.0

    # 1) Hard filter if user selected a doc type
    if filter_doc_type and filter_doc_type != "All":
        retriever = index.as_retriever(
            similarity_top_k=k,
            filters=MetadataFilters(filters=[
                MetadataFilter(key="doc_type", value=filter_doc_type, operator=FilterOperator.EQ)
            ])
        )
        routed_type = filter_doc_type
        confidence = 1.0
        # No Gemini call here
        nodes = retriever.retrieve(query)
        return nodes, routed_type, confidence

    # 2) Auto-route (Gemini) with caching
    if auto_route:
        cache_key = query.strip().lower()
        if cache_key in route_cache:
            routed_type, confidence = route_cache[cache_key]
            # print(f"  ♻️  Route cache hit: {routed_type} ({confidence:.2f})")
        else:
            routed_type, confidence = predict_doc_type_for_query(query)
            route_cache[cache_key] = (routed_type, confidence)
            # print(f"  🔀 Auto-routed to: {routed_type} (confidence: {confidence:.2f})")

        # 3) Decide whether to apply metadata filters
        # - If Gemini says "Other": do GLOBAL retrieval, but it's not "low confidence"
        # - If confidence is low: GLOBAL retrieval
        if routed_type != "Other" and confidence >= min_confidence_to_filter:
            retriever = index.as_retriever(
                similarity_top_k=k,
                filters=MetadataFilters(filters=[
                    MetadataFilter(key="doc_type", value=routed_type, operator=FilterOperator.EQ)
                ])
            )
        else:
            retriever = index.as_retriever(similarity_top_k=k)

        nodes = retriever.retrieve(query)
        return nodes, routed_type, confidence

    # 4) No filter + no routing
    retriever = index.as_retriever(similarity_top_k=k)
    nodes = retriever.retrieve(query)
    return nodes, None, 1.0


In [9]:
# ============================================================
# STEP 9: Answer Generation with Mistral + Source Attribution
# ============================================================
# Builds a focused prompt from retrieved chunks and sends it to Mistral.
# Returns a dict with answer text, source citations, and confidence.
# Adapted from Enhanced Q&A notebook's generate_answer_with_sources().

def generate_answer(
    query: str,
    nodes,
    routed_type: Optional[str] = None,
    confidence: float = 1.0
) -> Dict:
    """
    Generate an answer from retrieved nodes using Mistral.
    Returns dict: {answer, sources, confidence, filter_used}
    """
    if not nodes:
        return {
            "answer": "I couldn't find relevant information to answer your question. "
                      "Please ensure the PDF has been processed and try rephrasing your query.",
            "sources": [],
            "confidence": 0.0,
            "filter_used": "none"
        }

    # Build context string with provenance labels
    context_parts = []
    sources = []
    # ── Build context with smart trimming ────────────────────────────────────
    budget        = available_prompt_budget()
    system_prompt = (
        "[INST] You are a pharmaceutical document assistant specialising in quality, "
        "packaging, and compliance documentation. Use ONLY the provided context to answer "
        "the question. Be specific, cite the source file, document type, and page(s). "
        "If the context does not contain enough information, say so clearly.\n\nContext:\n"
    )
    question_block = f"\n\nQuestion: {query} [/INST]"
    overhead_tokens = estimate_tokens(system_prompt + question_block)
    context_budget  = budget - overhead_tokens

    # Fit as many chunks as possible within the token budget
    context_parts_trimmed = []
    tokens_used = 0
    chunks_dropped = 0

    for part in context_parts:
        part_tokens = estimate_tokens(part)
        if tokens_used + part_tokens <= context_budget:
            context_parts_trimmed.append(part)
            tokens_used += part_tokens
        else:
            chunks_dropped += 1

    if chunks_dropped > 0:
        print(f"  ⚠️  Context budget: {context_budget} tokens. "
              f"Dropped {chunks_dropped} chunk(s) to fit. "
              f"Try reducing 'Chunks to Retrieve' if answers seem incomplete.")

    context = "\n\n".join(context_parts_trimmed)
    prompt  = system_prompt + context + question_block
    total_prompt_tokens = estimate_tokens(prompt)

    # Hard safety check — should not trigger with the trimming above
    if total_prompt_tokens > budget:
        return {
            "answer": (
                f"⚠️ The retrieved context is too large for the current context window "
                f"({total_prompt_tokens} estimated tokens vs {budget} available). "
                f"Please reduce **Chunks to Retrieve** in the Retrieval Settings and try again."
            ),
            "sources":     sources,
            "confidence":  0.0,
            "filter_used": routed_type or "global",
            "chunks_used": len(context_parts_trimmed)
        }

def _approx_token_count(text: str) -> int:
    # crude but works: llama.cpp tokens are usually ~3-4 chars/token in English
    return max(1, len(text) // 4)

def build_rag_prompt_with_budget(
    query: str,
    nodes,
    *,
    max_ctx_tokens: int = 4096,
    max_new_tokens: int = 512,
    safety_margin_tokens: int = 128,
) -> Tuple[str, List[Dict], float]:
    """
    Build a prompt from retrieved nodes but enforce a context window budget.
    Returns: (prompt, sources, avg_score)

    This prevents: ValueError Requested tokens exceed context window.
    """

    if not nodes:
        return "", [], 0.0

    # Reserve room for the model to generate + formatting overhead
    # Available tokens for prompt text:
    budget = max_ctx_tokens - max_new_tokens - safety_margin_tokens
    if budget < 512:
        budget = 512  # emergency minimum

    header = (
        "[INST] You are a pharmaceutical document assistant specialising in quality, packaging, "
        "and compliance documentation.\n"
        "Use ONLY the provided context to answer the question.\n"
        "Be specific, cite the source file, document type, and page(s) where the information was found.\n"
        "If the context does not contain enough information, say so clearly.\n\n"
        "Context:\n"
    )

    footer = f"\n\nQuestion: {query} [/INST]"

    used = _approx_token_count(header) + _approx_token_count(footer)
    context_parts = []
    sources = []

    # Add nodes until we hit budget
    for node in nodes:
        meta = node.metadata or {}
        doc_type = meta.get("doc_type", "Unknown")
        p_start  = meta.get("page_start", "?")
        p_end    = meta.get("page_end", "?")
        source_f = meta.get("source_file", "Unknown File")
        score    = node.score if node.score is not None else 0.0

        chunk_text = node.text
        block = f"[Source File: {source_f} | Type: {doc_type} | Pages {p_start}–{p_end}]\n{chunk_text}\n"

        block_tokens = _approx_token_count(block)

        # If adding this block would exceed budget, stop
        if used + block_tokens > budget:
            break

        context_parts.append(block)
        used += block_tokens

        sources.append({
            "source_file": source_f,
            "doc_type": doc_type,
            "pages": f"{p_start}–{p_end}",
            "relevance": f"{score:.2%}" if score else "N/A",
            "preview": chunk_text[:120] + "..."
        })

    prompt = header + "\n".join(context_parts) + footer

    avg_score = sum(n.score for n in nodes if n.score is not None) / max(len(nodes), 1)
    return prompt, sources, avg_score




In [14]:
# ============================================================
# STEP 10: EnhancedDocumentStore — Central Orchestrator
# ============================================================
# Single object that holds all state for one Gradio session.
# Exposes process_pdf() and query() — the only two methods
# the UI callbacks need to call.

class EnhancedDocumentStore:
    """
    Orchestrates the full pipeline:
      upload → extract → classify → group → chunk → index → retrieve → answer
    """

    def __init__(self):
        self.pages:        List[PageInfo]       = []
        self.logical_docs: List[LogicalDocument] = []
        self.chunk_meta:   List[ChunkMetadata]  = []
        self.llama_docs:   List[Document]       = []
        self.index:        Optional[VectorStoreIndex] = None
        self.is_ready:     bool = False
        self.stats:        Dict = {}
        self.filename:     str = ""
        self.processed_files: set[str] = set()

    def _normalize_filepaths(self, pdf_files) -> List[str]:
        """Normalize Gradio file input into a list of file paths (strings)."""
        if not pdf_files:
            return []
        if isinstance(pdf_files, str):
            return [pdf_files]
        return list(pdf_files)

    def _get_new_files(self, filepaths: List[str]) -> List[str]:
        """Return only filepaths that have not been processed yet."""
        new_files = []
        for p in filepaths:
            fname = os.path.basename(p)
            if fname not in self.processed_files:
                new_files.append(p)
        return new_files


    # ── Public API ────────────────────────────────────────────────────

    def process_pdf(self, pdf_files) -> Tuple[bool, Dict]:
        """
        Full processing pipeline. Returns (success, stats_dict).
        Accepts a single filepath string or a list of filepaths.
        Incrementally process one or more PDFs and update the vector index without rebuilding it.

        Behavior:
        - Button-only usage: call this only when user clicks Process.
        - Processes only PDFs not seen before (by basename).
        - First ingestion builds the index; subsequent ingestions insert new nodes.
        """
        self.is_ready = False
        t0 = datetime.now()

        filepaths = self._normalize_filepaths(pdf_files)
        if not filepaths:
            return False, {"error": "No PDF files provided."}

        new_files = self._get_new_files(filepaths)
        if not new_files:
            # Nothing to do; keep system ready
            self.is_ready = self.index is not None
            return True, {**self.stats, "message": "No new files to process."}

        try:
            all_new_pages, all_new_logical_docs, all_new_chunk_meta, all_new_llama_docs = [], [], [], []

            for pdf_path in new_files:
                fname = os.path.basename(pdf_path)
                print(f"\n📄 Processing NEW file: {fname}")
                # 1. Extract + classify + group
                pages, logical_docs = extract_and_analyze_pdf(pdf_path, filename=fname)
                # 2. Chunk with metadata
                chunk_meta, llama_docs = process_all_documents(logical_docs)

                all_new_pages.extend(pages)
                all_new_logical_docs.extend(logical_docs)
                all_new_chunk_meta.extend(chunk_meta)
                all_new_llama_docs.extend(llama_docs)

                # mark processed
                self.processed_files.add(fname)

            # Append to global store (don’t replace)
            self.pages.extend(all_new_pages)
            self.logical_docs.extend(all_new_logical_docs)
            self.chunk_meta.extend(all_new_chunk_meta)
            self.llama_docs.extend(all_new_llama_docs)

            # Build or incrementally insert into the index
            if self.index is None:
                self.index = build_vector_index(all_new_llama_docs)
            else:
                # Incremental insertion (no rebuild)
                for d in all_new_llama_docs:
                    self.index.insert(d)

            elapsed = (datetime.now() - t0).total_seconds()

            # Recompute stats (cheap)
            self.stats = {
                "filename": f"{len(self.processed_files)} file(s)",
                "total_pages": len(self.pages),
                "documents_found": len(self.logical_docs),
                "total_chunks": len(self.chunk_meta),
                "document_types": list(dict.fromkeys(ld.doc_type for ld in self.logical_docs)),
                "processing_time": f"{elapsed:.1f}s",
            }

            self.is_ready = True
            return True, self.stats

        except Exception as e:
            import traceback; traceback.print_exc()
            return False, {"error": str(e)}


    def retrieve_only(
        self,
        question: str,
        filter_doc_type: Optional[str] = None,
        auto_route: bool = True,
        k: int = 4,
    ):
        """
        Retrieval only (no answer generation).
        Returns: (nodes, routed_type, confidence)
        """
        if not self.is_ready or self.index is None:
            return [], None, 0.0

        nodes, routed_type, confidence = retrieve_chunks(
            self.index,
            question,
            filter_doc_type=filter_doc_type,
            auto_route=auto_route,
            k=k,
        )
        return nodes, routed_type, confidence

    def query_stream(
        self,
        question: str,
        *,
        filter_doc_type: Optional[str] = None,
        auto_route: bool = True,
        k: int = 4,
        max_new_tokens: int = 512,
        temperature: float = 0.1,
        max_ctx_tokens: int = 4096,
    ):
        """
        Generator that yields partial assistant text (for Gradio streaming).
        """
        nodes, routed_type, confidence = self.retrieve_only(
            question,
            filter_doc_type=filter_doc_type,
            auto_route=auto_route,
            k=k,
        )

        if not nodes:
            yield {
                "answer_partial": "I couldn't find relevant information to answer your question. "
                                  "Please ensure the PDF has been processed and try rephrasing your query.",
                "sources": [],
                "filter_used": "none",
            }
            return

        prompt, sources, avg_score = build_rag_prompt_with_budget(
            question,
            nodes,
            max_ctx_tokens=max_ctx_tokens,
            max_new_tokens=max_new_tokens,
        )

        acc = ""
        for tok in stream_mistral(prompt, max_tokens=max_new_tokens, temperature=temperature):
            acc += tok
            yield {
                "answer_partial": acc,
                "sources": sources,
                "filter_used": routed_type or "global",
                "confidence": avg_score,
            }

    def query(
        self,
        question: str,
        filter_doc_type: Optional[str] = None,
        auto_route: bool = True,
        k: int = 4
    ) -> Dict:
        """Query the document store with a natural language question."""
        if not self.is_ready:
            return {
                "answer":     "Please upload and process a pharmaceutical PDF first.",
                "sources":    [],
                "confidence": 0.0,
                "filter_used": "none"
            }

        nodes, routed_type, confidence = retrieve_chunks(
            self.index, question,
            filter_doc_type=filter_doc_type,
            auto_route=auto_route,
            k=k
        )
        return generate_answer(question, nodes, routed_type, confidence)

    def get_document_structure(self) -> List[Dict]:
        """Return a summary list for display in the Document Info panel."""
        return [
            {
                "source_file": ld.source_file,
                "type":   ld.doc_type,
                "pages":  f"{ld.page_start}–{ld.page_end}",
                "chunks": len(ld.chunks) if ld.chunks else 0,
            }
            for ld in self.logical_docs
        ]


# Singleton — shared across all Gradio callbacks
doc_store = EnhancedDocumentStore()
print("✅ EnhancedDocumentStore initialised.")



✅ EnhancedDocumentStore initialised.


In [15]:
# ============================================================
# STEP 11: Gradio Interface
# ============================================================
# 3-column layout from the Enhanced Q&A notebook +
# Pfizer branding + Export button from Project 8.
#
# Column 1 (left):   PDF upload + Process / Clear buttons
# Column 2 (middle): Document Info panel + Retrieval Settings
# Column 3 (right):  Chat interface + example buttons
# Bottom row:        Status bar (Status | Documents | Chunks)

PFIZER_LOGO = (
    "https://upload.wikimedia.org/wikipedia/commons/0/0b/Pfizer_logo.svg"
)

def stream_mistral(prompt: str, *, max_tokens: int = 512, temperature: float = 0.1):
    """
    Streams token text from llama_cpp.Llama.create_completion(stream=True)
    """
    stream = mistral_llm.create_completion(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stream=True,
    )
    for chunk in stream:
        text = chunk["choices"][0].get("text", "")
        if text:
            yield text

def process_pdf_handler(pdf_files):
    """Called when user clicks 'Process Document' or uploads a new file.
        Handles one or multiple uploaded PDF files."""
    if not pdf_files:
        return (
            "Waiting for PDF upload...", "",
            gr.update(choices=["All"], value="All"),
            _status_bar_text()
        )

    # gr.File with file_count="multiple" returns a list of paths
    if isinstance(pdf_files, str):
        pdf_files = [pdf_files]
    # doc_store = EnhancedDocumentStore
    success, stats = doc_store.process_pdf(pdf_files)

    if success:
        status_md = (
            f"**Successfully Processed:**\n"
            f"- Files: {stats['filename']}\n"
            f"- Pages: {stats['total_pages']}\n"
            f"- Documents Found: {stats['documents_found']}\n"
            f"- Chunks Created: {stats['total_chunks']}\n"
            f"- Types: {', '.join(stats['document_types'])}\n"
            f"- Time: {stats['processing_time']}"
        )
        # structure = doc_store.get_document_structure()
        # structure_md = "\n".join(
        #     f"- **{d['type']}** (Pages {d['pages']}): {d['chunks']} chunks"
        #     for d in structure
        # )
        structure = doc_store.get_document_structure()

        # Group documents by their source file
        grouped_structure = {}
        for d in structure:
            fname = d['source_file']
            if fname not in grouped_structure:
                grouped_structure[fname] = []
            grouped_structure[fname].append(d)

        # Build the formatted Markdown string
        md_lines = []
        for fname, docs in grouped_structure.items():
            md_lines.append(f"\n**📄 {fname}**")
            for d in docs:
                md_lines.append(f"  - **{d['type']}** (Pages {d['pages']}) — {d['chunks']} chunks")

        structure_md = "\n".join(md_lines)

        type_choices = ["All"] + stats["document_types"]
        return (
            status_md, structure_md,
            gr.update(choices=type_choices, value="All"),
            _status_bar_text()
        )
    else:
        return (
            f"❌ Error: {stats.get('error', 'Unknown error')}", "",
            gr.update(choices=["All"], value="All"),
            _status_bar_text()
        )


def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    if history is None:
        history = []

    if not doc_store.is_ready:
        reply = ("Please upload and process a pharmaceutical PDF document first. "
                 "Use the **Upload Pharmaceutical PDF(s)** panel on the left, then click **Process Document**.")
        history = history + [{"role": "user", "content": message},
                             {"role": "assistant", "content": reply}]
        yield history, _status_bar_text()
        return

    if not message or not message.strip():
        yield history, _status_bar_text()
        return

    filter_type = None if (not doc_filter or doc_filter == "All") else doc_filter

    # add user message + empty assistant message for streaming updates
    history = history + [{"role": "user", "content": message},
                         {"role": "assistant", "content": ""}]
    yield history, _status_bar_text()

    final_sources = []
    final_filter_used = "global"
    acc = ""

    for partial in doc_store.query_stream(
        message,
        filter_doc_type=filter_type,
        auto_route=bool(auto_route) and filter_type is None,
        k=int(num_chunks),
        max_new_tokens=512,
        temperature=0.1,
        max_ctx_tokens=4096,
    ):
        acc = partial["answer_partial"]
        final_sources = partial.get("sources", [])
        final_filter_used = partial.get("filter_used", "global")

        history[-1]["content"] = acc
        yield history, _status_bar_text()

    # append sources once at the end
    if final_sources:
        citations = "\n\n**Sources:**\n" + "".join(
            f"- 📄 `{s['source_file']}` | {s['doc_type']} (Pages {s['pages']}) — Relevance: {s['relevance']}\n"
            for s in final_sources
        )
        history[-1]["content"] = acc.strip() + citations + f"\n\n*Filter: {final_filter_used}*"
        yield history, _status_bar_text()

    # export logging (optional)
    with open("chat_history.txt", "a", encoding="utf-8") as f:
        f.write(f"User: {message}\n")
        f.write(f"Assistant: {acc.strip()}\n")
        f.write("-" * 60 + "\n")


def _status_bar_text() -> str:
    if doc_store.is_ready:
        s = doc_store.stats
        return (f"**Status:** Ready | "
                f"**Documents:** {s.get('documents_found', 0)} | "
                f"**Chunks:** {s.get('total_chunks', 0)}")
    return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0"


def clear_all():
    global doc_store
    doc_store = EnhancedDocumentStore()
    # Clear the chat export file too
    open("chat_history.txt", "w").close()
    return (
        None,                                    # pdf_input
        "Waiting for PDF upload...",             # status_output
        "",                                      # structure_output
        gr.update(choices=["All"], value="All"), # doc_filter
        [],                                      # chatbot
        "",                                      # msg_input
        _status_bar_text()                       # status_bar
    )


# def make_example_handler(question):
#     """Factory for the example-question buttons."""
#     def _handler(history, doc_filter, auto_route, num_chunks):
#         return chat_handler(question, history, doc_filter, auto_route, num_chunks)
#     return _handler

def make_example_handler(question: str):
    """
    Gradio expects the function to be a generator yielding exactly the same outputs
    as chat_handler: (chatbot, status_bar).
    """
    def _handler(history, doc_filter, auto_route, num_chunks):
        # Delegate to the main streaming chat handler generator
        yield from chat_handler(question, history, doc_filter, auto_route, num_chunks)
    return _handler

def create_interface():
    with gr.Blocks(
        title="Pfizer Pharmaceutical Document RAG Chatbot",
        theme=gr.themes.Soft()
    ) as demo:

        # ── Header ──────────────────────────────────────────────────
        gr.HTML(f'''
        <div style="display:flex;align-items:center;margin-bottom:8px;">
            <img src="{PFIZER_LOGO}"
                 style="height:56px;margin-right:18px;" alt="Pfizer">
            <div>
                <h2 style="margin:0;font-size:1.5rem;">
                    Pfizer Pharmaceutical Document RAG Chatbot
                </h2>
                <p style="margin:2px 0 0;color:#555;font-size:0.85rem;">
                    Intelligent Multi-Document Analysis with Advanced RAG Pipeline
                    (Mistral-7B · BGE Embeddings · LlamaIndex)
                </p>
            </div>
        </div>
        <p style="color:#666;font-size:0.8rem;margin:0 0 4px;">
          Upload a pharmaceutical blob PDF (e.g. pharma-blob-sample.pdf) to identify
          document types, build a searchable index, and ask questions in natural language.
        </p>
        ''')

        with gr.Row():
            # ── Column 1: Upload ───────────────────────────────────
            with gr.Column(scale=2):
                pdf_input = gr.File(
                    label="Upload Pharmaceutical PDF(s)",
                    file_types=[".pdf"],
                    file_count="multiple",
                    type="filepath"
                )
                with gr.Row():
                    process_btn = gr.Button(
                        "Process Document", variant="primary", size="lg", scale=2
                    )
                    clear_all_btn = gr.Button(
                        "Clear All", variant="secondary", size="lg", scale=1
                    )

            # ── Column 2: Doc Info + Settings ─────────────────────
            with gr.Column(scale=1):
                gr.Markdown("### Document Info")
                status_output = gr.Markdown("Waiting for PDF upload...")
                structure_output = gr.Markdown("")

                gr.Markdown("### Retrieval Settings")
                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Document Type Filter",
                    info="Filter search to a specific pharmaceutical document type"
                )
                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Automatically detect the most relevant document type"
                )
                num_chunks = gr.Slider(
                    minimum=1, maximum=10, value=4, step=1,
                    label="Chunks to Retrieve"
                )

            # ── Column 3: Chat ─────────────────────────────────────
            with gr.Column(scale=2):
                gr.Markdown("### Ask Questions")
                chatbot = gr.Chatbot(
                    height=440,
                    show_label=False,
                    type="messages"
                )
                with gr.Row():
                    msg_input = gr.Textbox(
                        placeholder="e.g., What is the lot number? What sterilization method was used?",
                        show_label=False,
                        scale=4
                    )
                    send_btn = gr.Button("Send", variant="primary", scale=1)

                with gr.Row():
                    clear_chat_btn = gr.Button("Clear Chat", size="sm", scale=1)
                    sum_btn  = gr.Button("Summarise this document", size="sm", scale=1)
                    lot_btn  = gr.Button("Find lot numbers",        size="sm", scale=1)

        # ── Status Bar ─────────────────────────────────────────────
        with gr.Row():
            status_bar = gr.Markdown(_status_bar_text())

       # ── Export ─────────────────────────────────────────────────
        with gr.Row():
            export_btn  = gr.Button("📥 Export Chat History", scale=1)
            export_file = gr.File(label="Download", visible=False, scale=1)

        def export_chat_history():
            path = "/content/chat_history.txt"
            # Create the file if it doesn't exist yet
            if not os.path.exists(path):
                open(path, "w").close()
            return gr.update(value=path, visible=True)

        export_btn.click(fn=export_chat_history, outputs=[export_file])

        # ── Wiring ─────────────────────────────────────────────────
        proc_outputs = [status_output, structure_output, doc_filter, status_bar]

        process_btn.click(
            fn=process_pdf_handler,
            inputs=[pdf_input],
            outputs=proc_outputs
        )
        # pdf_input.change(
        #     fn=process_pdf_handler,
        #     inputs=[pdf_input],
        #     outputs=proc_outputs
        # )
        clear_all_btn.click(
            fn=clear_all,
            outputs=[pdf_input, status_output, structure_output,
                     doc_filter, chatbot, msg_input, status_bar]
        )

        chat_inputs  = [msg_input, chatbot, doc_filter, auto_route, num_chunks]
        chat_outputs = [chatbot, status_bar]

        send_btn.click(chat_handler, inputs=chat_inputs, outputs=chat_outputs
                       ).then(lambda: "", outputs=[msg_input])
        msg_input.submit(chat_handler, inputs=chat_inputs, outputs=chat_outputs
                         ).then(lambda: "", outputs=[msg_input])
        clear_chat_btn.click(lambda: [], outputs=[chatbot])

        # Example buttons
        sum_handler = make_example_handler(
            "Can you provide a summary of the main points in this document?"
        )
        lot_handler = make_example_handler(
            "What lot numbers or batch numbers are mentioned in these documents?"
        )
        sum_btn.click(sum_handler,
                      inputs=[chatbot, doc_filter, auto_route, num_chunks],
                      outputs=chat_outputs)
        lot_btn.click(lot_handler,
                      inputs=[chatbot, doc_filter, auto_route, num_chunks],
                      outputs=chat_outputs)

    return demo


In [ ]:
# ============================================================
# STEP 12: Launch the Application
# ============================================================
# share=True generates a public Gradio link valid for 72 hours.
# debug=True prints server-side logs to the cell output.

demo = create_interface()
demo.launch(share=True, debug=True)


/tmp/ipykernel_748/3594743835.py:196: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_748/3594743835.py:265: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://112abb9a73550e2665.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


📄 Processing NEW file: pharma-blob-sample.pdf
🔍 Starting PDF extraction and analysis for pharma-blob-sample.pdf...
  📄 Extracted 10 pages from PDF
🧠 Classifying pages and detecting document boundaries...
  ⚡ Rule-based: 10/10 pages classified instantly
  ✂️  Type-change boundaries (no API): 6
  🔍 Same-type transitions to check (parallel): 3


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2434.28ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5294.66ms


...

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

# 🧪 Pharmaceutical RAG Pipeline — Evaluation
### Pipeline Performance Metrics | Results from Initial Testing

Evaluates the Enhanced Mistral RAG Chatbot across three metric groups:

| Group | Metrics |
|---|---|
| 📋 Retrieval Performance | Recall@K, MRR, Precision@K, Hit Rate |
| 🍎 End-to-End Accuracy | Answer Accuracy, Citation Accuracy, Factual Consistency |
| 🤖 System Performance | Avg Response Time, Retrieval Latency, LLM Generation Time |


In [ ]:
# ============================================================
# STEP 1: Install Evaluation Dependencies
# ============================================================
!pip install -q rouge-score pandas tabulate
print("✅ Evaluation dependencies ready.")


  Preparing metadata (setup.py) ... done
✅ Evaluation dependencies ready.


In [ ]:
# ============================================================
# STEP 2: Imports
# ============================================================
import json, time, re
import pandas as pd
from tabulate import tabulate
from datetime import datetime
from typing import List, Dict, Optional

# Assumes the main RAG pipeline notebook is already run
# and doc_store is available in scope.
print("✅ Imports complete.")


✅ Imports complete.


In [ ]:
# ============================================================
# STEP 3: Ground Truth Definitions
# ============================================================
# Documents used:
#   1. Sigma-Aldrich CoA — Trizma® base (T1503, Lot 023H5602)
#   2. Sartorius BSE/TSE Declaration — Optifit / Safetyspace® tips
#
# Each case defines:
#   relevant_doc_types : list of doc types that SHOULD appear in top-K chunks
#   expected_keywords  : key terms the answer MUST contain (for factual check)
#   expected_citations : doc types that should be cited as sources
#   expected           : full reference answer for answer accuracy scoring

GROUND_TRUTH = [

    # ── Certificate of Analysis ───────────────────────────────────────────────
    {
        "id": "coa_01",
        "query": "What is the product name on the certificate of analysis?",
        "expected": "Trizma base",
        "expected_keywords": ["trizma", "base"],
        "relevant_doc_types": ["Certificate of Quality"],
        "expected_citations": ["Certificate of Quality"],
    },
    {
        "id": "coa_02",
        "query": "What is the lot number on the Sigma-Aldrich certificate?",
        "expected": "023H5602",
        "expected_keywords": ["023H5602"],
        "relevant_doc_types": ["Certificate of Quality"],
        "expected_citations": ["Certificate of Quality"],
    },
    {
        "id": "coa_03",
        "query": "What is the purity result for Trizma base?",
        "expected": "99.9%",
        "expected_keywords": ["99.9"],
        "relevant_doc_types": ["Certificate of Quality"],
        "expected_citations": ["Certificate of Quality"],
    },
    {
        "id": "coa_04",
        "query": "What is the water content result by Karl Fischer?",
        "expected": "0.03%",
        "expected_keywords": ["0.03"],
        "relevant_doc_types": ["Certificate of Quality"],
        "expected_citations": ["Certificate of Quality"],
    },
    {
        "id": "coa_05",
        "query": "What is the melting range of the product?",
        "expected": "171-173 DEG C",
        "expected_keywords": ["171", "173"],
        "relevant_doc_types": ["Certificate of Quality"],
        "expected_citations": ["Certificate of Quality"],
    },
    {
        "id": "coa_06",
        "query": "What is the heavy metals result?",
        "expected": "Less than 1 PPM as Lead",
        "expected_keywords": ["1", "ppm", "lead"],
        "relevant_doc_types": ["Certificate of Quality"],
        "expected_citations": ["Certificate of Quality"],
    },
    {
        "id": "coa_07",
        "query": "What is the CAS number for the product?",
        "expected": "77-86-1",
        "expected_keywords": ["77-86-1"],
        "relevant_doc_types": ["Certificate of Quality"],
        "expected_citations": ["Certificate of Quality"],
    },
    {
        "id": "coa_08",
        "query": "Who signed the certificate of analysis and what is their role?",
        "expected": "Rodney Burbach, Manager Quality Control, St. Louis, Missouri USA",
        "expected_keywords": ["burbach", "quality control"],
        "relevant_doc_types": ["Certificate of Quality"],
        "expected_citations": ["Certificate of Quality"],
    },

    # ── BSE/TSE Declaration ───────────────────────────────────────────────────
    {
        "id": "bse_01",
        "query": "Which products does the BSE/TSE declaration cover?",
        "expected": "Optifit Standard tips and Safetyspace Filtered pipette tips",
        "expected_keywords": ["optifit", "safetyspace"],
        "relevant_doc_types": ["BSE/TSE Declaration"],
        "expected_citations": ["BSE/TSE Declaration"],
    },
    {
        "id": "bse_02",
        "query": "Are any materials of animal origin used in these products?",
        "expected": "No, no materials of animal origin are used",
        "expected_keywords": ["no", "animal"],
        "relevant_doc_types": ["BSE/TSE Declaration"],
        "expected_citations": ["BSE/TSE Declaration"],
    },
    {
        "id": "bse_03",
        "query": "Which EU regulation is referenced in the BSE/TSE declaration?",
        "expected": "Commission Regulation (EC) No 722/2012",
        "expected_keywords": ["722/2012"],
        "relevant_doc_types": ["BSE/TSE Declaration"],
        "expected_citations": ["BSE/TSE Declaration"],
    },
    {
        "id": "bse_04",
        "query": "When was the BSE/TSE declaration signed?",
        "expected": "January 21st 2025",
        "expected_keywords": ["january", "2025"],
        "relevant_doc_types": ["BSE/TSE Declaration"],
        "expected_citations": ["BSE/TSE Declaration"],
    },
    {
        "id": "bse_05",
        "query": "Who are the signatories of the BSE/TSE declaration and their roles?",
        "expected": "Kirsi Jarvi EHS Manager and Tuomas Huhmarniemi Head of Quality",
        "expected_keywords": ["kirsi", "tuomas", "quality"],
        "relevant_doc_types": ["BSE/TSE Declaration"],
        "expected_citations": ["BSE/TSE Declaration"],
    },
    {
        "id": "bse_06",
        "query": "Who is the manufacturer in the BSE/TSE declaration?",
        "expected": "Sartorius Liquid Handling oy Helsinki Finland",
        "expected_keywords": ["sartorius", "helsinki"],
        "relevant_doc_types": ["BSE/TSE Declaration"],
        "expected_citations": ["BSE/TSE Declaration"],
    },

    # ── Cross-document ────────────────────────────────────────────────────────
    {
        "id": "cross_01",
        "query": "What quality standards or regulations are mentioned across these documents?",
        "expected": "ISO 9001 and EU Regulation EC No 722/2012",
        "expected_keywords": ["regulation", "quality"],
        "relevant_doc_types": ["Certificate of Quality", "BSE/TSE Declaration"],
        "expected_citations": ["Certificate of Quality", "BSE/TSE Declaration"],
    },

    # ── Robustness / Edge Cases ───────────────────────────────────────────────
    {
        "id": "edge_01",
        "query": "What is the expiration date of the product?",
        "expected": "NOT_IN_DOCUMENT",
        "expected_keywords": [],
        "relevant_doc_types": [],
        "expected_citations": [],
    },
    {
        "id": "edge_02",
        "query": "asjdhaksjdh random nonsense xyz query",
        "expected": "NOT_IN_DOCUMENT",
        "expected_keywords": [],
        "relevant_doc_types": [],
        "expected_citations": [],
    },
]

print(f"✅ Ground truth loaded: {len(GROUND_TRUTH)} test cases")


✅ Ground truth loaded: 17 test cases


In [ ]:
# ============================================================
# STEP 4: Metric Calculation Functions
# ============================================================

# ─────────────────────────────────────────────────────────────
# RETRIEVAL METRICS
# ─────────────────────────────────────────────────────────────

def precision_at_k(retrieved_types: List[str], relevant_types: List[str], k: int) -> float:
    """Precision@K: fraction of top-K retrieved chunks that are relevant."""
    if not relevant_types or not retrieved_types:
        return 1.0 if not relevant_types else 0.0
    top_k = retrieved_types[:k]
    hits = sum(1 for t in top_k if t in relevant_types)
    return round(hits / k, 3)


def recall_at_k(retrieved_types: List[str], relevant_types: List[str], k: int) -> float:
    """Recall@K: fraction of relevant doc types found in top-K chunks."""
    if not relevant_types:
        return 1.0
    top_k = retrieved_types[:k]
    found = set(t for t in top_k if t in relevant_types)
    return round(len(found) / len(set(relevant_types)), 3)


def reciprocal_rank(retrieved_types: List[str], relevant_types: List[str]) -> float:
    """Reciprocal Rank: 1/rank of first relevant chunk found."""
    if not relevant_types:
        return 1.0
    for rank, t in enumerate(retrieved_types, start=1):
        if t in relevant_types:
            return round(1.0 / rank, 3)
    return 0.0


def hit_rate(retrieved_types: List[str], relevant_types: List[str]) -> bool:
    """Hit Rate: True if at least 1 relevant doc type appears in retrieved chunks."""
    if not relevant_types:
        return True
    return any(t in relevant_types for t in retrieved_types)


# ─────────────────────────────────────────────────────────────
# END-TO-END ACCURACY METRICS
# ─────────────────────────────────────────────────────────────

def answer_accuracy(prediction: str, expected: str) -> bool:
    """
    Answer Accuracy: checks if all expected keywords appear in the prediction.
    For NOT_IN_DOCUMENT cases, checks the model admits it can't find the answer.
    """
    pred_lower = prediction.lower()

    if expected == "NOT_IN_DOCUMENT":
        refusal_phrases = [
            "not found", "not mentioned", "not available", "no information",
            "cannot find", "does not", "not in", "not provided", "i could not",
            "not contain", "unable to find"
        ]
        return any(p in pred_lower for p in refusal_phrases)

    exp_lower = expected.lower()
    key_terms = [t for t in exp_lower.split() if len(t) > 2]
    if not key_terms:
        return False
    matches = sum(1 for t in key_terms if t in pred_lower)
    return matches >= max(1, len(key_terms) // 2)


def citation_accuracy(sources: List[Dict], expected_citations: List[str]) -> float:
    """
    Citation Accuracy: fraction of expected doc types that appear in cited sources.
    Returns 1.0 if no citations expected (edge/robustness cases).
    """
    if not expected_citations:
        return 1.0
    if not sources:
        return 0.0
    cited_types = [s.get("doc_type", "") for s in sources]
    hits = sum(1 for exp in expected_citations if any(exp in ct for ct in cited_types))
    return round(hits / len(expected_citations), 3)


def factual_consistency(prediction: str, expected_keywords: List[str]) -> bool:
    """
    Factual Consistency: checks that all required factual terms appear in the answer.
    A missing required keyword suggests a hallucination or wrong retrieval.
    """
    if not expected_keywords:
        return True   # edge cases have no required keywords
    pred_lower = prediction.lower()
    return all(kw.lower() in pred_lower for kw in expected_keywords)


print("✅ Metric functions ready.")


✅ Metric functions ready.


In [ ]:
# ============================================================
# STEP 5: Run Full Evaluation
# ============================================================
# ⚠️  Ensure doc_store has processed the test PDFs before running.
#     Upload Sigma-Aldrich CoA + Sartorius BSE/TSE via the Gradio UI,
#     or uncomment the block below for programmatic processing:

doc_store = EnhancedDocumentStore()
print("✅ EnhancedDocumentStore initialised.")

success, stats = doc_store.process_pdf(
    ["/content/Sigma-Aldrich.pdf", "/content/bse_tse_sartorius.pdf"]
    # "/content/Sigma-Aldrich.pdf"
)
print(stats)

K = 4   # number of chunks retrieved per query

results = []
print(f"Running {len(GROUND_TRUTH)} test cases (K={K})...\n")

for case in GROUND_TRUTH:
    # ── Retrieval timing ──────────────────────────────────────
    t_retrieval_start = time.time()

    try:
        nodes, routed_type, confidence = retrieve_chunks(
            doc_store.index,
            case["query"],
            filter_doc_type=None,
            auto_route=True,
            k=K
        )
        retrieval_latency_ms = round((time.time() - t_retrieval_start) * 1000, 1)

        retrieved_types = [n.metadata.get("doc_type", "Unknown") for n in nodes]

        # ── LLM generation timing ─────────────────────────────
        t_llm_start = time.time()
        result      = generate_answer(case["query"], nodes, routed_type, confidence)
        llm_latency_s = round(time.time() - t_llm_start, 2)

        total_latency_s = round((retrieval_latency_ms / 1000) + llm_latency_s, 2)

        prediction = result.get("answer", "")
        print(prediction)
        sources    = result.get("sources", [])

        # ── Retrieval metrics ─────────────────────────────────
        p_at_k  = precision_at_k(retrieved_types, case["relevant_doc_types"], K)
        r_at_k  = recall_at_k(retrieved_types, case["relevant_doc_types"], K)
        rr      = reciprocal_rank(retrieved_types, case["relevant_doc_types"])
        hit     = hit_rate(retrieved_types, case["relevant_doc_types"])

        # ── End-to-end accuracy metrics ───────────────────────
        ans_acc  = answer_accuracy(prediction, case["expected"])
        cit_acc  = citation_accuracy(sources, case["expected_citations"])
        fact_con = factual_consistency(prediction, case["expected_keywords"])

        results.append({
            "id":                  case["id"],
            "query":               case["query"][:55] + "...",
            # Retrieval
            "precision_at_k":      p_at_k,
            "recall_at_k":         r_at_k,
            "reciprocal_rank":     rr,
            "hit_rate":            hit,
            # End-to-end
            "answer_accuracy":     ans_acc,
            "citation_accuracy":   cit_acc,
            "factual_consistency": fact_con,
            # Latency
            "retrieval_latency_ms": retrieval_latency_ms,
            "llm_latency_s":        llm_latency_s,
            "total_latency_s":      total_latency_s,
            "error": False
        })

        status = "✅" if ans_acc else "❌"
        print(f"{status} [{case['id']}] R@{K}={r_at_k} | MRR={rr} | Hit={hit} | "
              f"AnsAcc={ans_acc} | CitAcc={cit_acc} | Fact={fact_con} | {total_latency_s}s")

    except Exception as e:
        retrieval_latency_ms = round((time.time() - t_retrieval_start) * 1000, 1)
        results.append({
            "id": case["id"], "query": case["query"][:55] + "...",
            "precision_at_k": 0.0, "recall_at_k": 0.0,
            "reciprocal_rank": 0.0, "hit_rate": False,
            "answer_accuracy": False, "citation_accuracy": 0.0,
            "factual_consistency": False,
            "retrieval_latency_ms": retrieval_latency_ms,
            "llm_latency_s": 0.0, "total_latency_s": 0.0,
            "error": True
        })
        print(f"💥 [{case['id']}] ERROR: {e}")

print(f"\n✅ Evaluation complete.")


✅ EnhancedDocumentStore initialised.

📄 Processing: Sigma-Aldrich.pdf
🔍 Starting PDF extraction and analysis for Sigma-Aldrich.pdf...
  📄 Extracted 2 pages from PDF
🧠 Classifying pages and detecting document boundaries...
  Page 1: [NEW] Certificate of Quality
  📂 Identified 1 logical document(s):
     • Certificate of Quality (pages 1–2)
  ✂️  Certificate of Quality: 1 chunk(s)
  ✅ Total chunks: 1

📄 Processing: bse_tse_sartorius.pdf
🔍 Starting PDF extraction and analysis for bse_tse_sartorius.pdf...
  📄 Extracted 1 pages from PDF
🧠 Classifying pages and detecting document boundaries...
  Page 1: [NEW] BSE/TSE Declaration
  📂 Identified 1 logical document(s):
     • BSE/TSE Declaration (pages 1–1)
  ✂️  BSE/TSE Declaration: 1 chunk(s)
  ✅ Total chunks: 1
🗂️  Building vector index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

  ✅ Index ready with 2 nodes.
{'filename': '2 file(s)', 'total_pages': 3, 'documents_found': 2, 'total_chunks': 2, 'document_types': ['Certificate of Quality', 'BSE/TSE Declaration'], 'processing_time': '14.2s'}
Running 17 test cases (K=4)...

  🔀 Auto-routed to: Certificate of Quality (confidence: 0.95)


/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1256: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


Trizma® base

Reference: Certificate of Analysis, Pages 1-2. Specifically, look for the Product Name mentioned multiple times in the document.
✅ [coa_01] R@4=1.0 | MRR=1.0 | Hit=True | AnsAcc=True | CitAcc=1.0 | Fact=True | 9.24s
  🔀 Auto-routed to: Certificate of Quality (confidence: 0.98)
The lot number on the Sigma-Aldrich certificate is 023H5602. (Refer to the context on page 1 under the section "LOT 023H5602 RESULTS".)
✅ [coa_02] R@4=1.0 | MRR=1.0 | Hit=True | AnsAcc=True | CitAcc=1.0 | Fact=True | 4.84s
  🔀 Auto-routed to: Certificate of Quality (confidence: 0.95)
According to the provided Certificate of Analysis (COA) on pages 1-2, the purity result for Trizma base is 99.9%. (NLT 99.9% and 99.9% are stated under the "Purity by Sulfuric Acid Titration" section.)
✅ [coa_03] R@4=1.0 | MRR=1.0 | Hit=True | AnsAcc=True | CitAcc=1.0 | Fact=True | 4.65s
  🔀 Auto-routed to: Certificate of Quality (confidence: 0.98)
The water content results by Karl Fischer for Trizma® base, Lot 023H5602

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1254.64ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1003.60ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 802.89ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 928.35ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1831.25ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 23899.93ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3310.85ms


  🔀 Auto-routed to: Certificate of Quality (confidence: 0.95)
The melting range of the product, Trizma® base, is 171-173 degrees Celsius. (Certificate of Analysis, Page 2)
✅ [coa_05] R@4=1.0 | MRR=1.0 | Hit=True | AnsAcc=True | CitAcc=1.0 | Fact=True | 46.64s
  🔀 Auto-routed to: Certificate of Quality (confidence: 0.98)
The heavy metals result is less than 1 PPM (as lead.)(Certificate of Analysis, Lot 023H5602, Page 2)
✅ [coa_06] R@4=1.0 | MRR=1.0 | Hit=True | AnsAcc=True | CitAcc=1.0 | Fact=True | 4.62s


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4590.03ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 978.50ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 18407.50ms


  🔀 Auto-routed to: Certificate of Quality (confidence: 0.95)
The CAS number for the product is 77-86-1. (Certificate of Analysis, Page 1)
✅ [coa_07] R@4=1.0 | MRR=1.0 | Hit=True | AnsAcc=True | CitAcc=1.0 | Fact=True | 30.33s
  🔀 Auto-routed to: Certificate of Quality (confidence: 0.95)
The certificate of analysis provided in the context does not contain information about who signed it or what is the role of the signatory.
❌ [coa_08] R@4=1.0 | MRR=1.0 | Hit=True | AnsAcc=False | CitAcc=1.0 | Fact=False | 5.22s
  🔀 Auto-routed to: BSE/TSE Declaration (confidence: 0.95)
The BSE/TSE declaration covers the following products of Sartorius Liquid Handling oy: Optifit and Safetyspace®. (Refer to the context on Pages 1–1 of the BSE/TSE Declaration.)
❌ [bse_01] R@4=1.0 | MRR=1.0 | Hit=True | AnsAcc=False | CitAcc=1.0 | Fact=True | 8.71s
  🔀 Auto-routed to: BSE/TSE Declaration (confidence: 0.95)
According to the provided BSE/TSE Declaration on pages 1-1 from Sartorius Liquid Handling oy, the st

In [ ]:
# ============================================================
# STEP 6: Results Summary — Three Metric Groups
# ============================================================

df = pd.DataFrame(results)
total = len(df)

# ── Group 1: Retrieval Performance ───────────────────────────
n_hit      = df["hit_rate"].sum()
avg_p_at_k = round(df["precision_at_k"].mean() * 100, 1)
avg_r_at_k = round(df["recall_at_k"].mean() * 100, 1)
mrr        = round(df["reciprocal_rank"].mean(), 3)
hit_rate_pct = round(n_hit / total * 100, 1)

# ── Group 2: End-to-End Accuracy ─────────────────────────────
ans_acc_pct  = round(df["answer_accuracy"].sum() / total * 100, 1)
cit_acc_pct  = round(df["citation_accuracy"].mean() * 100, 1)
fact_con_pct = round(df["factual_consistency"].sum() / total * 100, 1)

# ── Group 3: System Performance ──────────────────────────────
avg_total_latency   = round(df["total_latency_s"].mean(), 2)
avg_retrieval_ms    = round(df["retrieval_latency_ms"].mean(), 1)
avg_llm_s           = round(df["llm_latency_s"].mean(), 2)
error_rate          = round(df["error"].sum() / total * 100, 1)

print("=" * 60)
print("  PIPELINE PERFORMANCE METRICS — Evaluation Results")
print("=" * 60)

print("\n📋 RETRIEVAL PERFORMANCE")
retrieval_table = [
    ["Metric",            "Value",          "Target"],
    [f"Recall@{K}",       f"{avg_r_at_k}%", "> 80%"],
    ["Mean Reciprocal Rank (MRR)", f"{mrr}", "> 0.70"],
    [f"Precision@{K}",    f"{avg_p_at_k}%", "> 70%"],
    ["Hit Rate",          f"{hit_rate_pct}%","100%"],
]
print(tabulate(retrieval_table, headers="firstrow", tablefmt="github"))

print("\n🍎 END-TO-END ACCURACY")
e2e_table = [
    ["Metric",               "Value",           "Evaluated On"],
    ["Answer Accuracy",      f"{ans_acc_pct}%",  f"{total} test questions"],
    ["Citation Accuracy",    f"{cit_acc_pct}%",  "correct source attribution"],
    ["Factual Consistency",  f"{fact_con_pct}%", "no hallucinations (keyword check)"],
]
print(tabulate(e2e_table, headers="firstrow", tablefmt="github"))

print("\n🤖 SYSTEM PERFORMANCE")
sys_table = [
    ["Metric",                     "Value"],
    ["Average Response Time",       f"{avg_total_latency}s"],
    ["Retrieval Latency",           f"{avg_retrieval_ms}ms"],
    ["LLM Generation Time (avg)",   f"{avg_llm_s}s"],
    ["Error Rate",                  f"{error_rate}%"],
]
print(tabulate(sys_table, headers="firstrow", tablefmt="github"))


  PIPELINE PERFORMANCE METRICS — Evaluation Results

📋 RETRIEVAL PERFORMANCE
| Metric                     | Value   | Target   |
|----------------------------|---------|----------|
| Recall@4                   | 100.0%  | > 80%    |
| Mean Reciprocal Rank (MRR) | 1.0     | > 0.70   |
| Precision@4                | 35.3%   | > 70%    |
| Hit Rate                   | 100.0%  | 100%     |

🍎 END-TO-END ACCURACY
| Metric              | Value   | Evaluated On                      |
|---------------------|---------|-----------------------------------|
| Answer Accuracy     | 88.2%   | 17 test questions                 |
| Citation Accuracy   | 100.0%  | correct source attribution        |
| Factual Consistency | 88.2%   | no hallucinations (keyword check) |

🤖 SYSTEM PERFORMANCE
| Metric                    | Value    |
|---------------------------|----------|
| Average Response Time     | 11.08s   |
| Retrieval Latency         | 7052.0ms |
| LLM Generation Time (avg) | 4.03s    |
| Error Rat

In [ ]:
# ============================================================
# STEP 7: Per-Query Detail Table
# ============================================================

print("\n=== Per-Query Results ===")
detail_cols = [
    "id", "precision_at_k", "recall_at_k", "reciprocal_rank",
    "hit_rate", "answer_accuracy", "citation_accuracy",
    "factual_consistency", "total_latency_s"
]
print(tabulate(df[detail_cols], headers="keys", tablefmt="github", showindex=False))



=== Per-Query Results ===
| id       |   precision_at_k |   recall_at_k |   reciprocal_rank | hit_rate   | answer_accuracy   |   citation_accuracy | factual_consistency   |   total_latency_s |
|----------|------------------|---------------|-------------------|------------|-------------------|---------------------|-----------------------|-------------------|
| coa_01   |             0.25 |             1 |                 1 | True       | True              |                   1 | True                  |              9.24 |
| coa_02   |             0.25 |             1 |                 1 | True       | True              |                   1 | True                  |              4.84 |
| coa_03   |             0.25 |             1 |                 1 | True       | True              |                   1 | True                  |              4.65 |
| coa_04   |             0.25 |             1 |                 1 | True       | True              |                   1 | True           

In [ ]:
# ============================================================
# STEP 8: Export Results
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path  = f"/content/eval_results_{timestamp}.csv"
json_path = f"/content/eval_summary_{timestamp}.json"

df.to_csv(csv_path, index=False)

summary = {
    "timestamp":           timestamp,
    "total_cases":         total,
    "K":                   K,
    "retrieval": {
        f"recall_at_{K}":      f"{avg_r_at_k}%",
        "mrr":                 mrr,
        f"precision_at_{K}":   f"{avg_p_at_k}%",
        "hit_rate":            f"{hit_rate_pct}%",
    },
    "end_to_end": {
        "answer_accuracy":     f"{ans_acc_pct}%",
        "citation_accuracy":   f"{cit_acc_pct}%",
        "factual_consistency": f"{fact_con_pct}%",
    },
    "system": {
        "avg_response_time_s":    avg_total_latency,
        "avg_retrieval_latency_ms": avg_retrieval_ms,
        "avg_llm_generation_s":   avg_llm_s,
        "error_rate":             f"{error_rate}%",
    }
}

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"✅ Saved: {csv_path}")
print(f"✅ Saved: {json_path}")
print(json.dumps(summary, indent=2))


✅ Saved: /content/eval_results_20260511_005226.csv
✅ Saved: /content/eval_summary_20260511_005226.json
{
  "timestamp": "20260511_005226",
  "total_cases": 17,
  "K": 4,
  "retrieval": {
    "recall_at_4": "100.0%",
    "mrr": 1.0,
    "precision_at_4": "35.3%",
    "hit_rate": "100.0%"
  },
  "end_to_end": {
    "answer_accuracy": "88.2%",
    "citation_accuracy": "100.0%",
    "factual_consistency": "88.2%"
  },
  "system": {
    "avg_response_time_s": 11.08,
    "avg_retrieval_latency_ms": 7052.0,
    "avg_llm_generation_s": 4.03,
    "error_rate": "0.0%"
  }
}


In [ ]:
queries = ["Which EU regulation is referenced in the BSE/TSE declaration?", "What quality standards or regulations are mentioned across these documents?"]

doc_store = EnhancedDocumentStore()
print("✅ EnhancedDocumentStore initialised.")
success, stats = doc_store.process_pdf([
                                        #"/content/pharma-blob-sample.pdf",
                                        "/content/Sigma-Aldrich.pdf",
                                        "/content/bse_tse_sartorius.pdf"
                                        ]
)
print(stats)
for query in queries:


    # 1. Retrieve the chunks
    nodes, routed_type, confidence = retrieve_chunks(doc_store.index, query)

    print(f"Query: {query}")
    print(f"Auto-Routed To: {routed_type} (Confidence: {confidence})\n")

    print("Retrieved Chunks:")
    for i, node in enumerate(nodes):
        # LlamaIndex stores the similarity calculation right here:
        score = node.score if node.score is not None else 0.0
        print(f"{i+1}. \"{node.text[:]}...\" (Similarity: {score:.2f})")

    # 2. Generate the final answer
    result = generate_answer(query, nodes, routed_type, confidence)
    print(f"\nFinal Answer: {result['answer']}")
    print('\n')

del doc_store

✅ EnhancedDocumentStore initialised.

📄 Processing: Sigma-Aldrich.pdf
🔍 Starting PDF extraction and analysis for Sigma-Aldrich.pdf...
  📄 Extracted 2 pages from PDF
🧠 Classifying pages and detecting document boundaries...
  Page 1: [NEW] Certificate of Quality
  Page 2: [NEW] Other
  📂 Identified 2 logical document(s):
     • Certificate of Quality (pages 1–1)
     • Other (pages 2–2)
  ✂️  Certificate of Quality: 1 chunk(s)
  ✂️  Other: 1 chunk(s)
  ✅ Total chunks: 2

📄 Processing: bse_tse_sartorius.pdf
🔍 Starting PDF extraction and analysis for bse_tse_sartorius.pdf...
  📄 Extracted 1 pages from PDF
🧠 Classifying pages and detecting document boundaries...
  Page 1: [NEW] BSE/TSE Declaration
  📂 Identified 1 logical document(s):
     • BSE/TSE Declaration (pages 1–1)
  ✂️  BSE/TSE Declaration: 1 chunk(s)
  ✅ Total chunks: 1
🗂️  Building vector index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

  ✅ Index ready with 3 nodes.
{'filename': '2 file(s)', 'total_pages': 3, 'documents_found': 3, 'total_chunks': 3, 'document_types': ['Certificate of Quality', 'Other', 'BSE/TSE Declaration'], 'processing_time': '39.9s'}
  🔀 Auto-routed to: BSE/TSE Declaration (confidence: 0.95)
Query: Which EU regulation is referenced in the BSE/TSE declaration?
Auto-Routed To: BSE/TSE Declaration (Confidence: 0.95)

Retrieved Chunks:
1. "Sartorius Liquid Handling oy 
Tulppatie 1, 00880 Helsinki, Finland 
lhinfo@sartorius.com 
sartorius.com 
Business ID: FI24418858 
Animal Origin and BSE/TSE Statement 
 
This certificate applies to following products: 
 
 
Optifit | Standard tips 
 
Safetyspace® | Filtered pipette tips 
 
 
Manufacturer: 
Sartorius Liquid Handling oy 
Tulppatie 1, 00880 Helsinki, Finland 
 
 
Transmissible spongiform encephalopathies (TSEs) are chronic degenerative nervous diseases 
characterised by the accumulation of an abnormal isoform of a cellular glycoprotein known as 
prion pro

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1269: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(



Final Answer: The EU regulation referenced in the BSE/TSE declaration is Commission Regulation (EC) No 722/2012. (Source File: bse_tse_sartorius.pdf | Type: BSE/TSE Declaration | Page 3, second paragraph)


  🔀 Auto-routed to: Other (confidence: 0.95)
  ⚠️  Low confidence → global search
Query: What quality standards or regulations are mentioned across these documents?
Auto-Routed To: Other (Confidence: 0.95)

Retrieved Chunks:
1. "Calculators & Apps
Webinars
Offices
therapy development and
production.
Sigma-
Aldrich®
Solutions
BioReliance®
Solutions
Millipore®
Solutions
SAFC®
Solutions
Milli-Q®
Solutions
Supelco®
Solutions
© 2026 Merck KGaA, Darmstadt, Germany and/or its affiliates. All Rights Reserved, including Text and Data Mining for AI training and similar technologies.
Reproduction of any materials from the site is strictly forbidden
without permission.
Site Use Terms | Privacy Policy | General Terms and Conditions of Sale | Copyright Consent
| Site Map | Cookie Settings..." (S